In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv("python_vulnerability_dataset.csv")
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, shuffle=True)


In [ ]:
df = pd.read_csv("python_vulnerability_dataset.csv")
df.info()

In [ ]:


import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ============ CUDA niceties ============
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
use_cuda = torch.cuda.is_available()
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ampere/Ada
use_tf32 = use_cuda and cap_major >= 8                                      # Ampere/Ada

if use_cuda:
    try: torch.set_float32_matmul_precision("high")  # has effect only where supported
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"CUDA visible  | GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | tf32_enabled={use_tf32}")

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary_dedup.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/codebert-base"
OUT_DIR     = "/codeptm_binary_codebert"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH, usecols=[TEXT_COL, LABEL_COL]).dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL])
train_df, val_df  = train_test_split(train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL])

pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)
ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)

# Windows-safe: avoid multiprocess tokenization to prevent pickling issues
num_proc = 1 if platform.system() == "Windows" else max(1, (os.cpu_count() or 4)//2)
ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL], num_proc=num_proc)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# model.gradient_checkpointing_enable()  # optional memory saver

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
        "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap, "mse": mse, "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train (GPU-friendly) ========
optim_choice = "adamw_torch_fused" if use_cuda else "adamw_torch"

args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",      # correct key
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    bf16=use_bf16,                              # bf16 on Ampere/Ada
    fp16=(use_cuda and not use_bf16),           # else fp16 on GPU
    tf32=use_tf32,                              # TF32 only on Ampere/Ada
    optim=optim_choice,
    dataloader_pin_memory=True,
    dataloader_num_workers=max(2, (os.cpu_count() or 4) - 2),
    dataloader_persistent_workers=(platform.system() != "Windows"),
    seed=SEED,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_codebert_binary")
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Saved model ->", MODEL_SAVE_DIR)



In [ ]:
import os, json, numpy as np, pandas as pd, torch
from datasets import Dataset, DatasetDict, Sequence, Value
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

MODEL_NAME = "microsoft/codebert-base"        # or graphcodebert/unixcoder
MAX_LEN    = 512
SEED       = 42

# === 1) Load & preprocess ===
df = pd.read_csv("python_vulnerability_dataset.csv")

def parse_labels(s):
    if pd.isna(s) or not str(s).strip():
        return []
    return [t.strip() for t in str(s).split(",") if t.strip()]

# change this if your label column is named differently
LABEL_COL = "predicted_cwe_ids"
TEXT_COL  = "vulnerable_function_source"

df["labels_list"] = df[LABEL_COL].apply(parse_labels)

# Build label space
all_labels = sorted({lab for labs in df["labels_list"] for lab in labs})
if not all_labels:
    raise ValueError("No labels found. Check your label column and formatting.")
label2id = {l:i for i,l in enumerate(all_labels)}
id2label = {i:l for l,i in label2id.items()}

# Binarize (multi-hot)
mlb = MultiLabelBinarizer(classes=all_labels)
Y = mlb.fit_transform(df["labels_list"])

# Split (stratify by 'has any label' to avoid empty test/train)
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=(Y.sum(axis=1) > 0)
)
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
Y_train, Y_test   = Y[train_idx], Y[test_idx]

# Small validation from train for threshold tuning
train_idx2, val_idx = train_test_split(
    np.arange(len(train_df)), test_size=0.1, random_state=SEED,
    stratify=(Y_train.sum(axis=1) > 0)
)
val_df   = train_df.iloc[val_idx].copy()
Y_val    = Y_train[val_idx]
train_df = train_df.iloc[train_idx2].copy()
Y_train  = Y_train[train_idx2]

# Attach multi-hot arrays (as lists of floats to satisfy BCEWithLogitsLoss)
train_df["multi_hot"] = [list(map(float, row)) for row in Y_train]
val_df["multi_hot"]   = [list(map(float, row)) for row in Y_val]
test_df["multi_hot"]  = [list(map(float, row)) for row in Y_test]

# HF datasets
train_ds = Dataset.from_pandas(train_df[[TEXT_COL, "multi_hot"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[[TEXT_COL, "multi_hot"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[[TEXT_COL, "multi_hot"]], preserve_index=False)
ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)

# === 2) Tokenize ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    # ensure labels are floats (not ints)
    enc["labels"] = [list(map(float, x)) for x in batch["multi_hot"]]
    return enc

ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, "multi_hot"])

# Enforce float32 for the 'labels' column (prevents BCEWithLogits dtype crash)
for split in ds.keys():
    ds[split] = ds[split].cast_column("labels", Sequence(Value("float32")))

# Collator that guarantees float32 labels at batch-time (extra safety)
base_collator = DataCollatorWithPadding(tokenizer)
def float_label_collator(features):
    import torch
    labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)
    batch = base_collator([{k:v for k,v in f.items() if k!="labels"} for f in features])
    batch["labels"] = labels
    return batch

# === 3) Model ===
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(all_labels)
)
model.config.problem_type = "multi_label_classification"
model.config.id2label = id2label
model.config.label2id = label2id

# === 4) Metrics (with threshold tuning on val) ===
THRESHOLD = 0.5

def binarize(probs, thr=THRESHOLD):
    return (probs >= thr).astype(int)

def compute_scores(probs, labels, thr):
    y_pred = binarize(probs, thr)
    scores = {
        "f1_micro":  f1_score(labels, y_pred, average="micro", zero_division=0),
        "f1_macro":  f1_score(labels, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(labels, y_pred, average="micro", zero_division=0),
        "recall_micro":    recall_score(labels, y_pred, average="micro", zero_division=0),
    }
    try:
        scores["auc_micro"] = roc_auc_score(labels, probs, average="micro")
    except Exception:
        scores["auc_micro"] = float("nan")
    return scores

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1/(1+np.exp(-logits))
    return compute_scores(probs, labels, THRESHOLD)

# === 5) Train ===
args = TrainingArguments(
    output_dir="./codeptm_multilabel",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",      # <-- fixed key
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),   # avoid fp16 on CPU
    gradient_accumulation_steps=2,
    logging_steps=50,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=float_label_collator,   # <— ensures float labels
    compute_metrics=compute_metrics,
)

trainer.train()

# === 6) Threshold tuning on validation ===
val_logits = trainer.predict(ds["validation"]).predictions
val_probs  = 1/(1+np.exp(-val_logits))
val_labels = np.stack(ds["validation"]["labels"])  # already float32

best_thr, best_f1 = 0.5, -1
for thr in [i/100 for i in range(25, 76)]:  # 0.25 .. 0.75
    f1 = f1_score(val_labels, (val_probs>=thr).astype(int), average="micro", zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f"Best validation micro-F1 @ threshold={best_thr:.2f}: {best_f1:.4f}")
THRESHOLD = best_thr

# === 7) Test ===
test_logits = trainer.predict(ds["test"]).predictions
test_probs  = 1/(1+np.exp(-test_logits))
test_labels = np.stack(ds["test"]["labels"])
test_scores = compute_scores(test_probs, test_labels, THRESHOLD)
print("TEST:", test_scores)

# === 8) Save model + label map + threshold ===
trainer.save_model("./codeptm_multilabel/best")
with open("./codeptm_multilabel/labels.json","w") as f:
    json.dump({"label2id":label2id, "id2label":id2label, "threshold":THRESHOLD}, f, indent=2)


In [ ]:
import numpy as np

# expects: test_labels (0/1 array), test_probs (float probs), THRESHOLD (float)
def simple_metrics(y_true, y_prob, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= thr).astype(int)

    tp = np.logical_and(y_true == 1, y_pred == 1).sum()
    tn = np.logical_and(y_true == 0, y_pred == 0).sum()
    fp = np.logical_and(y_true == 0, y_pred == 1).sum()
    fn = np.logical_and(y_true == 1, y_pred == 0).sum()

    total = tp + tn + fp + fn
    acc = (tp + tn) / max(total, 1)

    sens_den = tp + fn
    spec_den = tn + fp
    sensitivity = tp / sens_den if sens_den > 0 else float("nan")
    specificity = tn / spec_den if spec_den > 0 else float("nan")

    return {
        "accuracy_micro": float(acc),
        "sensitivity_micro": float(sensitivity),
        "specificity_micro": float(specificity),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
        "threshold": float(thr),
    }

metrics = simple_metrics(test_labels, test_probs, THRESHOLD)
print(metrics)


In [ ]:
# !pip install -q transformers datasets scikit-learn accelerate

import os, json, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/codebert-base"  # swap to graphcodebert/unixcoder if you like
OUT_DIR     = "/content/drive/MyDrive/Python_data/codeptm_binary_codebert"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH)

# keep only needed cols, drop na, enforce label type
df = df[[TEXT_COL, LABEL_COL]].dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL]
)

train_csv = os.path.join(OUT_DIR, "train.csv")
test_csv  = os.path.join(OUT_DIR, "test.csv")
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Saved splits -> {train_csv} , {test_csv}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(
        batch[TEXT_COL],
        truncation=True, max_length=MAX_LEN, padding=False
    )
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

# create a small validation split from train (10%) for early/better eval
train_val = train_ds.train_test_split(test_size=0.10, seed=SEED, stratify_by_column=LABEL_COL)
ds = DatasetDict(train=train_val["train"], validation=train_val["test"], test=test_ds)

ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL])
collator = DataCollatorWithPadding(tokenizer)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2  # binary with CrossEntropyLoss
)

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity (TPR)
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    # specificity (TNR)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,              # sensitivity
        "sensitivity": rec,         # alias
        "specificity": spec,
        "f1": f1,
        "mcc": mcc,
        "kappa": kap,
        "mse": mse,
        "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train ========
args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_codebert_binary")
trainer.save_model(MODEL_SAVE_DIR)        # saves config + model weights
tokenizer.save_pretrained(MODEL_SAVE_DIR) # saves tokenizer files
print("Saved model ->", MODEL_SAVE_DIR)

# ======== 8) (Optional) Save per-sample predictions on test set ========
import torch
model.eval()
probs_list, preds_list = [], []
for i in range(0, len(ds["test"]), 32):
    batch = ds["test"][i:i+32]
    enc = {k: torch.tensor(v).to(model.device) for k, v in batch.items() if k in ["input_ids","attention_mask","token_type_ids"]}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    probs_list.append(probs)
    preds_list.append(np.argmax(probs, axis=1))

probs = np.vstack(probs_list)
preds = np.concatenate(preds_list)
test_out = test_df.copy().reset_index(drop=True)
test_out["pred_label"] = preds
test_out["prob_non_vuln"] = probs[:,0]
test_out["prob_vuln"] = probs[:,1]
preds_csv = os.path.join(OUT_DIR, "test_predictions.csv")
test_out.to_csv(preds_csv, index=False)
print("Saved predictions ->", preds_csv)


In [ ]:
# !pip install -q transformers datasets scikit-learn accelerate

import os, json, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/codebert-base"
OUT_DIR     = "/codeptm_binary_codebert"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH)[[TEXT_COL, LABEL_COL]].dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL]
)

# make a 10% validation FROM TRAIN (stratified)
train_df, val_df = train_test_split(
    train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL]
)

# save splits
pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)
ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL])

collator = DataCollatorWithPadding(tokenizer)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    # specificity (TNR) with guards
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    else:
        tn = fp = fn = tp = 0
        spec = float("nan")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,              # sensitivity
        "sensitivity": rec,         # alias
        "specificity": spec,
        "f1": f1,
        "mcc": mcc,
        "kappa": kap,
        "mse": mse,
        "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train ========
args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_codebert_binary")
trainer.save_model(MODEL_SAVE_DIR)        # saves config + model weights
tokenizer.save_pretrained(MODEL_SAVE_DIR) # saves tokenizer files
print("Saved model ->", MODEL_SAVE_DIR)

# ======== 8) Save per-sample predictions on test set ========
probs_list, preds_list = [], []
model.eval()
for i in range(0, len(ds["test"]), 32):
    batch = ds["test"][i:i+32]
    enc = {k: torch.tensor(v).to(model.device)
           for k, v in batch.items()
           if k in ["input_ids", "attention_mask", "token_type_ids"] and v is not None}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    probs_list.append(probs)
    preds_list.append(np.argmax(probs, axis=1))

probs = np.vstack(probs_list)
preds = np.concatenate(preds_list)

test_out = test_df.copy().reset_index(drop=True)
test_out["pred_label"] = preds
test_out["prob_non_vuln"] = probs[:, 0]
test_out["prob_vuln"] = probs[:, 1]
preds_csv = os.path.join(OUT_DIR, "test_predictions.csv")
test_out.to_csv(preds_csv, index=False)
print("Saved predictions ->", preds_csv)


In [ ]:


import os, json, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ============ CUDA niceties ============
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
if torch.cuda.is_available():
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print("CUDA visible ✅  |  GPU:", torch.cuda.get_device_name(0))

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/codebert-base"
OUT_DIR     = "/codeptm_binary_codebert"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# Prefer bf16 on Ampere/Ada (e.g., RTX 4090); else fp16
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.get_device_capability(0)[0] >= 8

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH)[[TEXT_COL, LABEL_COL]].dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL]
)
train_df, val_df = train_test_split(
    train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL]
)

pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)
ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL])

collator = DataCollatorWithPadding(tokenizer)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# Optional memory tradeoff:
# model.gradient_checkpointing_enable()

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    else:
        tn = fp = fn = tp = 0
        spec = float("nan")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,              # sensitivity
        "sensitivity": rec,         # alias
        "specificity": spec,
        "f1": f1,
        "mcc": mcc,
        "kappa": kap,
        "mse": mse,
        "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train ========
args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",      # <-- fix key
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    # CUDA-friendly options:
    bf16=use_bf16,
    fp16=(use_cuda and not use_bf16),
    torch_compile=use_cuda,           # PyTorch 2.x, JIT-compile for speed
    dataloader_pin_memory=True,
    dataloader_num_workers=2,         # bump if CPU has more cores
    seed=SEED,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_codebert_binary")
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Saved model ->", MODEL_SAVE_DIR)

# ======== 8) Save per-sample predictions on test set (with AMP on GPU) ========
probs_list, preds_list = [], []
model.eval()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

for i in range(0, len(ds["test"]), 32):
    batch = ds["test"][i:i+32]
    enc = {k: torch.tensor(v).to(model.device)
           for k, v in batch.items()
           if k in ["input_ids", "attention_mask", "token_type_ids"] and v is not None}
    with torch.cuda.amp.autocast(enabled=use_cuda, dtype=amp_dtype):
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
    probs_list.append(probs)
    preds_list.append(np.argmax(probs, axis=1))

probs = np.vstack(probs_list)
preds = np.concatenate(preds_list)

test_out = test_df.copy().reset_index(drop=True)
test_out["pred_label"] = preds
test_out["prob_non_vuln"] = probs[:, 0]
test_out["prob_vuln"] = probs[:, 1]
preds_csv = os.path.join(OUT_DIR, "test_predictions.csv")
test_out.to_csv(preds_csv, index=False)
print("Saved predictions ->", preds_csv)


In [ ]:


import os, json, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ============ CUDA niceties ============
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
if torch.cuda.is_available():
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print("CUDA visible ✅  |  GPU:", torch.cuda.get_device_name(0))

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary_dedup.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/codebert-base"
OUT_DIR     = "/codeptm_binary_codebert_up"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# Prefer bf16 on Ampere/Ada (e.g., RTX 4090); else fp16
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.get_device_capability(0)[0] >= 8

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH)[[TEXT_COL, LABEL_COL]].dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL]
)
train_df, val_df = train_test_split(
    train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL]
)

pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)
ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL])

collator = DataCollatorWithPadding(tokenizer)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# Optional memory tradeoff:
# model.gradient_checkpointing_enable()

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    else:
        tn = fp = fn = tp = 0
        spec = float("nan")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,              # sensitivity
        "sensitivity": rec,         # alias
        "specificity": spec,
        "f1": f1,
        "mcc": mcc,
        "kappa": kap,
        "mse": mse,
        "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train ========
args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",      # <-- fix key
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    # CUDA-friendly options:

    fp16=True,
    torch_compile=use_cuda,           # PyTorch 2.x, JIT-compile for speed
    dataloader_pin_memory=True,
    dataloader_num_workers=2,         # bump if CPU has more cores
    seed=SEED,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_codebert_binary_up")
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Saved model ->", MODEL_SAVE_DIR)

# ======== 8) Save per-sample predictions on test set (with AMP on GPU) ========
probs_list, preds_list = [], []
model.eval()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

for i in range(0, len(ds["test"]), 32):
    batch = ds["test"][i:i+32]
    enc = {k: torch.tensor(v).to(model.device)
           for k, v in batch.items()
           if k in ["input_ids", "attention_mask", "token_type_ids"] and v is not None}
    with torch.cuda.amp.autocast(enabled=use_cuda, dtype=amp_dtype):
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
    probs_list.append(probs)
    preds_list.append(np.argmax(probs, axis=1))

probs = np.vstack(probs_list)
preds = np.concatenate(preds_list)

test_out = test_df.copy().reset_index(drop=True)
test_out["pred_label"] = preds
test_out["prob_non_vuln"] = probs[:, 0]
test_out["prob_vuln"] = probs[:, 1]
preds_csv = os.path.join(OUT_DIR, "test_predictions.csv")
test_out.to_csv(preds_csv, index=False)
print("Saved predictions ->", preds_csv)


In [ ]:
# !pip install -q transformers datasets scikit-learn accelerate

import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ============ CUDA niceties ============
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
use_cuda = torch.cuda.is_available()
if use_cuda:
    try: torch.set_float32_matmul_precision("high")  # TF32 on Ampere/Ada
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print("CUDA visible ✅  |  GPU:", torch.cuda.get_device_name(0))

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary_dedup.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/codebert-base"
OUT_DIR     = "/codeptm_binary_codebert_up"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# Prefer bf16 on Ampere/Ada; else fp16
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH, usecols=[TEXT_COL, LABEL_COL]).dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL]
)
train_df, val_df = train_test_split(
    train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL]
)

pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)

# ---- Windows-safe: no multiprocessing (avoids NameError: tokenizer not defined)
num_proc = 1 if platform.system() == "Windows" else max(1, (os.cpu_count() or 4) // 2)

ds = ds.map(
    tok, batched=True,
    remove_columns=[TEXT_COL, LABEL_COL],
    num_proc=num_proc
)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# model.gradient_checkpointing_enable()  # optional memory tradeoff

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)    # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return {"accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
            "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap, "mse": mse, "mae": mae,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

# ======== 5) Train (GPU-friendly) ========
optim_choice = "adamw_torch_fused" if use_cuda else "adamw_torch"

args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    bf16=use_bf16,
    fp16=(use_cuda and not use_bf16),
    tf32=True,                       # speed on Ampere/Ada
    optim=optim_choice,
    dataloader_pin_memory=True,
    dataloader_num_workers=max(2, (os.cpu_count() or 4) - 2),
    dataloader_persistent_workers=(platform.system() != "Windows"),
    seed=SEED,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_codebert_binary_up")
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Saved model ->", MODEL_SAVE_DIR)

# ======== 8) Save per-sample predictions (AMP on GPU) ========
probs_list, preds_list = [], []
model.eval()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

for i in range(0, len(ds["test"]), 64):
    batch = ds["test"][i:i+64]
    enc = {k: torch.tensor(v).to(model.device)
           for k, v in batch.items()
           if k in ["input_ids","attention_mask","token_type_ids"] and v is not None}
    with torch.cuda.amp.autocast(enabled=use_cuda, dtype=amp_dtype):
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
    probs_list.append(probs)
    preds_list.append(np.argmax(probs, axis=1))

probs = np.vstack(probs_list); preds = np.concatenate(preds_list)
out_df = test_df.reset_index(drop=True).copy()
out_df["pred_label"] = preds
out_df["prob_non_vuln"] = probs[:,0]
out_df["prob_vuln"] = probs[:,1]
out_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)
print("Saved predictions ->", os.path.join(OUT_DIR, "test_predictions.csv"))


In [ ]:
# !pip install -q transformers datasets scikit-learn accelerate

import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ============ CUDA niceties ============
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
use_cuda = torch.cuda.is_available()
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ampere/Ada
use_tf32 = use_cuda and cap_major >= 8                                      # Ampere/Ada

if use_cuda:
    try: torch.set_float32_matmul_precision("high")  # has effect only where supported
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"CUDA visible  | GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | tf32_enabled={use_tf32}")

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary_dedup.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/graphcodebert-base"
OUT_DIR     = "/codeptm_binary_graphcodebert"
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH, usecols=[TEXT_COL, LABEL_COL]).dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL])
train_df, val_df  = train_test_split(train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL])

pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)
ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)

# Windows-safe: avoid multiprocess tokenization to prevent pickling issues
num_proc = 1 if platform.system() == "Windows" else max(1, (os.cpu_count() or 4)//2)
ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL], num_proc=num_proc)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# model.gradient_checkpointing_enable()  # optional memory saver

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
        "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap, "mse": mse, "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train (GPU-friendly) ========
optim_choice = "adamw_torch_fused" if use_cuda else "adamw_torch"

args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",      # correct key
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    bf16=use_bf16,                              # bf16 on Ampere/Ada
    fp16=(use_cuda and not use_bf16),           # else fp16 on GPU
    tf32=use_tf32,                              # TF32 only on Ampere/Ada
    optim=optim_choice,
    dataloader_pin_memory=True,
    dataloader_num_workers=max(2, (os.cpu_count() or 4) - 2),
    dataloader_persistent_workers=(platform.system() != "Windows"),
    seed=SEED,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_graphcodebert_binary")
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Saved model ->", MODEL_SAVE_DIR)



In [ ]:
import pandas as pd

data = pd.read_csv('Reveal_vultest.csv')
data.info()

In [ ]:
data['output'].value_counts()

In [ ]:

import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR   = "/codeptm_binary_graphcodebert/model_graphcodebert_binary"
DEFAULT_TEST = "Reveal_vultest.csv"
TEST_CSV    = DEFAULT_TEST

TEXT_COL    = "input"
LABEL_COL   = "output"
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()  # <-- FIX: safe no-op context on CPU

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # <-- FIX: cast logits to float32 before .cpu().numpy() to avoid bfloat16 NumPy error
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


Codeptm_Unixcoder_finetuning

In [ ]:


import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

# ============ CUDA niceties ============
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
use_cuda = torch.cuda.is_available()
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ampere/Ada (e.g., RTX 4080)
use_tf32 = use_cuda and cap_major >= 8

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"CUDA visible  | GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | tf32_enabled={use_tf32}")

# ======== CONFIG ========
CSV_PATH    = "python_dataset_binary_dedup.csv"
TEXT_COL    = "vulnerable_function_source"
LABEL_COL   = "label"            # 0/1
MODEL_NAME  = "microsoft/unixcoder-base"        # <--- UniXcoder here
OUT_DIR     = "/codeptm_binary_unixcoder"       # <--- new output dir
EPOCHS      = 5
LR          = 2e-5
BATCH_TRAIN = 4
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

os.makedirs(OUT_DIR, exist_ok=True)

# ======== 1) Load & split (80/20) and save splits ========
df = pd.read_csv(CSV_PATH, usecols=[TEXT_COL, LABEL_COL]).dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)

train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df[LABEL_COL])
train_df, val_df  = train_test_split(train_df, test_size=0.10, random_state=SEED, stratify=train_df[LABEL_COL])

pd.concat([train_df, val_df]).to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)  # (train+val)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")

# ======== 2) Tokenizer & HF datasets ========
# UniXcoder works best with the python (slow) tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# Make sure a pad token exists for batching
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})

def tok(batch):
    enc = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)
ds = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)

num_proc = 1 if platform.system() == "Windows" else max(1, (os.cpu_count() or 4)//2)
ds = ds.map(tok, batched=True, remove_columns=[TEXT_COL, LABEL_COL], num_proc=num_proc)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

# ======== 3) Model ========
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# If we added a pad token above, resize embeddings to avoid index errors
model.resize_token_embeddings(len(tokenizer))

# Optional memory saver
# model.gradient_checkpointing_enable()

# ======== 4) Metrics ========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)        # positive=1
    rec  = recall_score(labels, preds, zero_division=0)           # sensitivity
    f1   = f1_score(labels, preds, zero_division=0)
    mcc  = matthews_corrcoef(labels, preds)
    kap  = cohen_kappa_score(labels, preds)
    mse  = mean_squared_error(labels, preds)
    mae  = mean_absolute_error(labels, preds)

    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
        "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap, "mse": mse, "mae": mae,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

# ======== 5) Train (GPU-friendly) ========
optim_choice = "adamw_torch_fused" if use_cuda else "adamw_torch"

args = TrainingArguments(
    output_dir=os.path.join(OUT_DIR, "hf_runs"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    bf16=use_bf16,                              # bf16 on Ada (e.g., 4080)
    fp16=(use_cuda and not use_bf16),           # else fp16 on GPU
    tf32=use_tf32,                              # TF32 only on Ampere/Ada
    optim=optim_choice,
    dataloader_pin_memory=True,
    dataloader_num_workers=max(2, (os.cpu_count() or 4) - 2),
    dataloader_persistent_workers=(platform.system() != "Windows"),
    seed=SEED,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ======== 6) Final test evaluation & save metrics ========
test_metrics = trainer.evaluate(ds["test"])
metrics_path = os.path.join(OUT_DIR, "test_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved metrics ->", metrics_path)
print(json.dumps(test_metrics, indent=2))

# ======== 7) Save model & tokenizer for later testing ========
MODEL_SAVE_DIR = os.path.join(OUT_DIR, "model_unixcoder_binary")
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print("Saved model ->", MODEL_SAVE_DIR)


hugging face data

In [ ]:
from datasets import load_dataset

ds = load_dataset("maddyrucos/code_vulnerability_python")

In [ ]:
# pip install -q datasets pandas

from datasets import load_dataset
import os

OUT_DIR = "./code_vulnerability_python_csv"
os.makedirs(OUT_DIR, exist_ok=True)

ds = load_dataset("maddyrucos/code_vulnerability_python")  # train-only dataset
print(ds)

# Save the single split to CSV
out_csv = os.path.join(OUT_DIR, "train.csv")
try:
    ds["train"].to_csv(out_csv)          # fast path (Hugging Face Datasets)
except Exception:
    ds["train"].to_pandas().to_csv(out_csv, index=False)  # fallback via pandas
print("Saved ->", out_csv)

# (optional) Save with your typical column names
# ds["train"].to_pandas().rename(
#     {"func": "vulnerable_function_source", "target": "label"}, axis=1
# ).to_csv(os.path.join(OUT_DIR, "train_renamed.csv"), index=False)


In [ ]:
# ======= Inference & Evaluation on Saved Model =======
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR   = "/codeptm_binary_graphcodebert/model_graphcodebert_binary"
DEFAULT_TEST = out_csv
TEST_CSV    = DEFAULT_TEST

TEXT_COL    = "func"
LABEL_COL   = "target"
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()  # <-- FIX: safe no-op context on CPU

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # <-- FIX: cast logits to float32 before .cpu().numpy() to avoid bfloat16 NumPy error
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


unixCODER

In [ ]:
# ======= Inference & Evaluation on Saved Model =======
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR   = "/codeptm_binary_unixcoder\model_unixcoder_binary"
DEFAULT_TEST = out_csv
TEST_CSV    = DEFAULT_TEST

TEXT_COL    = "func"
LABEL_COL   = "target"
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()  # <-- FIX: safe no-op context on CPU

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # <-- FIX: cast logits to float32 before .cpu().numpy() to avoid bfloat16 NumPy error
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


VUDENC

In [ ]:
from datasets import load_dataset

ds = load_dataset("DetectVul/Vudenc")

In [ ]:
# pip install -q datasets pandas

from datasets import load_dataset
import os

OUT_DIR = "./Vcode_vulnerability_python_csv"
os.makedirs(OUT_DIR, exist_ok=True)

ds = load_dataset("DetectVul/Vudenc")  # train-only dataset
print(ds)

# Save the single split to CSV
Vout_csv = os.path.join(OUT_DIR, "train.csv")
try:
    ds["train"].to_csv(out_csv)          # fast path (Hugging Face Datasets)
except Exception:
    ds["train"].to_pandas().to_csv(out_csv, index=False)  # fallback via pandas
print("Saved ->", out_csv)

# (optional) Save with your typical column names
# ds["train"].to_pandas().rename(
#     {"func": "vulnerable_function_source", "target": "label"}, axis=1
# ).to_csv(os.path.join(OUT_DIR, "train_renamed.csv"), index=False)


In [ ]:
# pip install -q datasets pandas

from datasets import load_dataset
import os, json

OUT_DIR = "./Vudenc_csv"
os.makedirs(OUT_DIR, exist_ok=True)

ds = load_dataset("DetectVul/Vudenc")  # has 'train' and 'test'

def save_split(split_name):
    d = ds[split_name]
    # Flatten list columns to JSON strings so each row is 1 sample
    def flatten(ex):
        return {
            "lines": json.dumps(ex["lines"], ensure_ascii=False),
            "raw_lines": json.dumps(ex["raw_lines"], ensure_ascii=False),
            "label": json.dumps(ex["label"], ensure_ascii=False),
            "type":  json.dumps(ex["type"],  ensure_ascii=False),
        }

    flat = d.map(flatten, remove_columns=d.column_names)
    path = os.path.join(OUT_DIR, f"Vudenc_{split_name}.csv")
    try:
        flat.to_csv(path)  # fast path
    except Exception:
        flat.to_pandas().to_csv(path, index=False)  # fallback
    print(f"Saved -> {path}  | rows={len(d)}")

for split in ("train", "test"):
    if split in ds:
        save_split(split)


Unix

In [ ]:
# ======= Inference & Evaluation on Saved Model =======
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR   = "/codeptm_binary_unixcoder\model_unixcoder_binary"
DEFAULT_TEST = "./Vudenc_csv\Vudenc_test.csv"
TEST_CSV    = DEFAULT_TEST

TEXT_COL    = "lines"
LABEL_COL   = "label"
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()  # <-- FIX: safe no-op context on CPU

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # <-- FIX: cast logits to float32 before .cpu().numpy() to avoid bfloat16 NumPy error
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


In [ ]:
# ======= Inference & Evaluation on Saved Model (Vudenc JSON columns) =======
import os, json, ast, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR    = r"/codeptm_binary_unixcoder/model_unixcoder_binary"   # use raw string or forward slashes
DEFAULT_TEST = r"./Vudenc_csv/Vudenc_test.csv"
TEST_CSV     = DEFAULT_TEST

# Columns in the CSV:
LINES_COL    = "lines"   # JSON array of strings
LABEL_COL    = "label"   # JSON array of 0/1 per line

# Aggregation for list labels -> single binary:
#   "any"      : 1 if any line is 1
#   "majority" : 1 if mean >= 0.5
#   "threshold": 1 if mean >= LABEL_THRESHOLD
LABEL_AGG        = "any"
LABEL_THRESHOLD  = 0.5

BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Helpers to parse JSONish strings safely ====
def parse_jsonish(x):
    if isinstance(x, (list, dict)):
        return x
    try:
        return json.loads(x)
    except Exception:
        try:
            return ast.literal_eval(x)
        except Exception:
            return x  # fallback (will handle later)

def aggregate_label(lbl_list):
    # Convert list-like to list of ints
    if not isinstance(lbl_list, list):
        return int(lbl_list)  # already scalar
    if len(lbl_list) == 0:
        return 0
    arr = np.array(lbl_list, dtype=float)
    mean_val = float(arr.mean())
    if LABEL_AGG == "any":
        return int(np.any(arr > 0.5))
    elif LABEL_AGG == "majority":
        return int(mean_val >= 0.5)
    elif LABEL_AGG == "threshold":
        return int(mean_val >= LABEL_THRESHOLD)
    else:
        return int(np.any(arr > 0.5))  # default

# ==== Load & prepare data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[LINES_COL, LABEL_COL]).copy()

# Parse JSONish columns
df[LINES_COL] = df[LINES_COL].apply(parse_jsonish)
df[LABEL_COL] = df[LABEL_COL].apply(parse_jsonish)

# Join code lines to a single string
df["text"] = df[LINES_COL].apply(lambda x: "\n".join(x) if isinstance(x, list) else str(x))

# Aggregate per-line labels to a single binary label
df["label_bin"] = df[LABEL_COL].apply(aggregate_label).astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(text, truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, "text", "label_bin", tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # Cast logits to float32 to avoid bfloat16 → NumPy issue
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds  = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs  = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics (binary, after aggregation) ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


In [ ]:
# ======= Inference & Evaluation on Saved Model (Vudenc JSON columns) =======
import os, json, ast, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR    = r"/codeptm_binary_unixcoder/model_unixcoder_binary"   # use raw string or forward slashes
DEFAULT_TEST = r"./Vudenc_csv/Vudenc_train.csv"
TEST_CSV     = DEFAULT_TEST

# Columns in the CSV:
LINES_COL    = "lines"   # JSON array of strings
LABEL_COL    = "label"   # JSON array of 0/1 per line

# Aggregation for list labels -> single binary:
#   "any"      : 1 if any line is 1
#   "majority" : 1 if mean >= 0.5
#   "threshold": 1 if mean >= LABEL_THRESHOLD
LABEL_AGG        = "any"
LABEL_THRESHOLD  = 0.5

BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Helpers to parse JSONish strings safely ====
def parse_jsonish(x):
    if isinstance(x, (list, dict)):
        return x
    try:
        return json.loads(x)
    except Exception:
        try:
            return ast.literal_eval(x)
        except Exception:
            return x  # fallback (will handle later)

def aggregate_label(lbl_list):
    # Convert list-like to list of ints
    if not isinstance(lbl_list, list):
        return int(lbl_list)  # already scalar
    if len(lbl_list) == 0:
        return 0
    arr = np.array(lbl_list, dtype=float)
    mean_val = float(arr.mean())
    if LABEL_AGG == "any":
        return int(np.any(arr > 0.5))
    elif LABEL_AGG == "majority":
        return int(mean_val >= 0.5)
    elif LABEL_AGG == "threshold":
        return int(mean_val >= LABEL_THRESHOLD)
    else:
        return int(np.any(arr > 0.5))  # default

# ==== Load & prepare data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[LINES_COL, LABEL_COL]).copy()

# Parse JSONish columns
df[LINES_COL] = df[LINES_COL].apply(parse_jsonish)
df[LABEL_COL] = df[LABEL_COL].apply(parse_jsonish)

# Join code lines to a single string
df["text"] = df[LINES_COL].apply(lambda x: "\n".join(x) if isinstance(x, list) else str(x))

# Aggregate per-line labels to a single binary label
df["label_bin"] = df[LABEL_COL].apply(aggregate_label).astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(text, truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, "text", "label_bin", tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # Cast logits to float32 to avoid bfloat16 → NumPy issue
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds  = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs  = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics (binary, after aggregation) ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


In [ ]:
ds = pd.read_csv("py.csv")
ds.info()

In [ ]:
ds['label'].value_counts()

In [ ]:
# ======= Inference & Evaluation on Saved Model (Vudenc JSON columns) =======
import os, json, ast, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR    = r"/codeptm_binary_graphcodebert/model_graphcodebert_binary"   # use raw string or forward slashes
DEFAULT_TEST = r"./Vudenc_csv/Vudenc_test.csv"
TEST_CSV     = DEFAULT_TEST

# Columns in the CSV:
LINES_COL    = "lines"   # JSON array of strings
LABEL_COL    = "label"   # JSON array of 0/1 per line

# Aggregation for list labels -> single binary:
#   "any"      : 1 if any line is 1
#   "majority" : 1 if mean >= 0.5
#   "threshold": 1 if mean >= LABEL_THRESHOLD
LABEL_AGG        = "any"
LABEL_THRESHOLD  = 0.5

BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Helpers to parse JSONish strings safely ====
def parse_jsonish(x):
    if isinstance(x, (list, dict)):
        return x
    try:
        return json.loads(x)
    except Exception:
        try:
            return ast.literal_eval(x)
        except Exception:
            return x  # fallback (will handle later)

def aggregate_label(lbl_list):
    # Convert list-like to list of ints
    if not isinstance(lbl_list, list):
        return int(lbl_list)  # already scalar
    if len(lbl_list) == 0:
        return 0
    arr = np.array(lbl_list, dtype=float)
    mean_val = float(arr.mean())
    if LABEL_AGG == "any":
        return int(np.any(arr > 0.5))
    elif LABEL_AGG == "majority":
        return int(mean_val >= 0.5)
    elif LABEL_AGG == "threshold":
        return int(mean_val >= LABEL_THRESHOLD)
    else:
        return int(np.any(arr > 0.5))  # default

# ==== Load & prepare data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[LINES_COL, LABEL_COL]).copy()

# Parse JSONish columns
df[LINES_COL] = df[LINES_COL].apply(parse_jsonish)
df[LABEL_COL] = df[LABEL_COL].apply(parse_jsonish)

# Join code lines to a single string
df["text"] = df[LINES_COL].apply(lambda x: "\n".join(x) if isinstance(x, list) else str(x))

# Aggregate per-line labels to a single binary label
df["label_bin"] = df[LABEL_COL].apply(aggregate_label).astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(text, truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, "text", "label_bin", tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # Cast logits to float32 to avoid bfloat16 → NumPy issue
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds  = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs  = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics (binary, after aggregation) ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


py.csv

In [ ]:
import pandas as pd

In [ ]:
df =pd.read_csv("py.csv")

In [ ]:
df.info()

In [ ]:

df['label'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import re

# 1) normalize label text (lowercase, trim, fix a couple typos you mentioned)
typo_fix = {
    "vulnerbale": "vulnerable",
    "nonvulnerblke": "non-vulnerable",
}
lab = (df['label']
       .astype(str).str.strip().str.lower()
       .replace(typo_fix))

# 2) map to binary with robust patterns
def to_bin(s: str):
    s = s.strip().lower()
    # exact numeric/boolean
    if s in {"1", "true", "yes"}:  return 1
    if s in {"0", "false", "no"}:  return 0
    # canonical words
    if s in {"vulnerable", "vuln", "vul"}: return 1
    if s in {"non-vulnerable", "nonvulnerable", "non_vulnerable", "benign", "safe", "clean", "secure"}: return 0
    # regex fallbacks (anchored so 'nonvulnerable' won't be mistaken as vulnerable)
    if re.fullmatch(r'(vuln(?:erable)?|v)', s): return 1
    if re.fullmatch(r'(non[-_ ]?vuln(?:erable)?|not[-_ ]?vuln(?:erable)?)', s): return 0
    return np.nan  # unknown -> NaN so you can inspect

df['label'] = lab.map(to_bin)  # or lab.apply(to_bin)

# Optional: if you want strict ints and no NaNs, drop or fill unknowns first:
# df = df.dropna(subset=['label'])      # drop unknowns
# df['label'] = df['label'].astype(int) # make sure it's int 0/1

print(df['label'].value_counts(dropna=False))


In [ ]:
# ======= Inference & Evaluation on Saved Model =======
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR   = "/codeptm_binary_unixcoder\model_unixcoder_binary"
DEFAULT_TEST = df
TEST_CSV    = DEFAULT_TEST

TEXT_COL    = "code"
LABEL_COL   = "label"
BATCH_EVAL  = 8
SEED        = 42
MAX_LEN     = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()  # Ada supports bf16
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()  # <-- FIX: safe no-op context on CPU

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)

            # <-- FIX: cast logits to float32 before .cpu().numpy() to avoid bfloat16 NumPy error
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


In [ ]:
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR    = "/codeptm_binary_graphcodebert/model_graphcodebert_binary"
DEFAULT_TEST = "py.csv"   # <-- updated
TEST_CSV     = DEFAULT_TEST

TEXT_COL     = "input"    # preferred name (will auto-fallback if missing)
LABEL_COL    = "output"   # preferred name (will auto-fallback if missing)

BATCH_EVAL   = 8
SEED         = 42
MAX_LEN      = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).copy()

# Auto-detect text/label columns if preferred names aren't present
if TEXT_COL not in df.columns:
    for cand in ["input", "text", "code", "lines", "content", "func", "function",
                 "code_snippet", "code_snip", "source", "body"]:
        if cand in df.columns:
            TEXT_COL = cand
            break
if LABEL_COL not in df.columns:
    for cand in ["output", "label", "labels", "target", "vulnerable", "class", "y"]:
        if cand in df.columns:
            LABEL_COL = cand
            break

missing_cols = [c for c in [TEXT_COL, LABEL_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Could not find required columns in CSV. Missing: {missing_cols}")

# Drop rows with empty text/label
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# ==== Map labels: vulnerable -> 1, non-vulnerable -> 0 (robust) ====
def map_label(x):
    s = str(x).strip().lower()
    # numeric/boolean passthroughs
    if s in {"1", "true", "yes"}:  return 1
    if s in {"0", "false", "no"}:  return 0

    # common typos/variants
    if s in {"vulnerbale", "vuln", "vul"}: return 1
    if s in {
        "non-vulnerable", "nonvulnerable", "non_vulnerable", "non vuln", "nonvuln",
        "benign", "safe", "clean", "secure", "not vulnerable"
    }: return 0

    # heuristic fallbacks (ordered to avoid matching 'non-vulnerable' as vulnerable)
    if "non" in s and "vul" in s: return 0
    if "vul" in s: return 1

    # last resort: try casting to int
    try:
        v = int(float(s))
        if v in (0, 1): return v
    except Exception:
        pass

    return np.nan  # unknown -> NaN so you can inspect/drop

# apply mapping if not already clean ints
if not np.issubdtype(df[LABEL_COL].dtype, np.number):
    df[LABEL_COL] = df[LABEL_COL].apply(map_label)
else:
    # ensure only 0/1; anything else becomes NaN then dropped
    df[LABEL_COL] = df[LABEL_COL].apply(lambda v: v if v in (0, 1) else np.nan)

# drop rows with unknown labels after mapping
before_n = len(df)
df = df.dropna(subset=[LABEL_COL]).copy()
after_n = len(df)
if after_n < before_n:
    print(f"Dropped {before_n - after_n} rows with unmapped/invalid labels.")

# cast to int
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("Resolved columns ->", {"text_col": TEXT_COL, "label_col": LABEL_COL})
print("Label distribution (0=non-vulnerable, 1=vulnerable):")
print(df[LABEL_COL].value_counts(dropna=False).sort_index())

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


In [ ]:
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR    = "/codeptm_binary_graphcodebert/model_graphcodebert_binary"
DEFAULT_TEST = "py.csv"   # <-- updated
TEST_CSV     = DEFAULT_TEST

TEXT_COL     = "input"    # preferred name (will auto-fallback if missing)
LABEL_COL    = "output"   # preferred name (will auto-fallback if missing)

BATCH_EVAL   = 8
SEED         = 42
MAX_LEN      = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).copy()

# Auto-detect text/label columns if preferred names aren't present
if TEXT_COL not in df.columns:
    for cand in ["input", "text", "code", "lines", "content", "func", "function",
                 "code_snippet", "code_snip", "source", "body"]:
        if cand in df.columns:
            TEXT_COL = cand
            break
if LABEL_COL not in df.columns:
    for cand in ["output", "label", "labels", "target", "vulnerable", "class", "y"]:
        if cand in df.columns:
            LABEL_COL = cand
            break

missing_cols = [c for c in [TEXT_COL, LABEL_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Could not find required columns in CSV. Missing: {missing_cols}")

# Drop rows with empty text/label
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# ==== Map labels: vulnerable -> 1, non-vulnerable -> 0 (robust) ====
def map_label(x):
    s = str(x).strip().lower()
    # numeric/boolean passthroughs
    if s in {"1", "true", "yes"}:  return 1
    if s in {"0", "false", "no"}:  return 0

    # common typos/variants
    if s in {"vulnerbale", "vuln", "vul"}: return 1
    if s in {
        "non-vulnerable", "nonvulnerable", "non_vulnerable", "non vuln", "nonvuln",
        "benign", "safe", "clean", "secure", "not vulnerable"
    }: return 0

    # heuristic fallbacks (ordered to avoid matching 'non-vulnerable' as vulnerable)
    if "non" in s and "vul" in s: return 0
    if "vul" in s: return 1

    # last resort: try casting to int
    try:
        v = int(float(s))
        if v in (0, 1): return v
    except Exception:
        pass

    return np.nan  # unknown -> NaN so you can inspect/drop

# apply mapping if not already clean ints
if not np.issubdtype(df[LABEL_COL].dtype, np.number):
    df[LABEL_COL] = df[LABEL_COL].apply(map_label)
else:
    # ensure only 0/1; anything else becomes NaN then dropped
    df[LABEL_COL] = df[LABEL_COL].apply(lambda v: v if v in (0, 1) else np.nan)

# drop rows with unknown labels after mapping
before_n = len(df)
df = df.dropna(subset=[LABEL_COL]).copy()
after_n = len(df)
if after_n < before_n:
    print(f"Dropped {before_n - after_n} rows with unmapped/invalid labels.")

# cast to int
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("Resolved columns ->", {"text_col": TEXT_COL, "label_col": LABEL_COL})
print("Label distribution (0=non-vulnerable, 1=vulnerable):")
print(df[LABEL_COL].value_counts(dropna=False).sort_index())

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")


In [ ]:
/codeptm_binary_unixcoder/model_unixcoder_binary

In [ ]:
import os, json, platform, numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error,
    confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from contextlib import nullcontext

# ==== CONFIG ====
MODEL_DIR    = "/codeptm_binary_unixcoder/model_unixcoder_binary"
DEFAULT_TEST = "py.csv"   # <-- updated
TEST_CSV     = DEFAULT_TEST

TEXT_COL     = "input"    # preferred name (will auto-fallback if missing)
LABEL_COL    = "output"   # preferred name (will auto-fallback if missing)

BATCH_EVAL   = 8
SEED         = 42
MAX_LEN      = 512

# Where to save evaluation artifacts
EVAL_OUT = os.path.join(MODEL_DIR, "eval_saved")
os.makedirs(EVAL_OUT, exist_ok=True)

# ==== Repro ====
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==== Device / dtypes ====
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
cap_major = torch.cuda.get_device_capability(0)[0] if use_cuda else 0
use_bf16 = use_cuda and cap_major >= 8 and torch.cuda.is_bf16_supported()
use_fp16 = use_cuda and not use_bf16

if use_cuda:
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | CC: {torch.cuda.get_device_capability(0)}")
    print(f"bf16_supported={use_bf16} | fp16_used={use_fp16}")

# ==== Load data ====
df = pd.read_csv(TEST_CSV).copy()

# Auto-detect text/label columns if preferred names aren't present
if TEXT_COL not in df.columns:
    for cand in ["input", "text", "code", "lines", "content", "func", "function",
                 "code_snippet", "code_snip", "source", "body"]:
        if cand in df.columns:
            TEXT_COL = cand
            break
if LABEL_COL not in df.columns:
    for cand in ["output", "label", "labels", "target", "vulnerable", "class", "y"]:
        if cand in df.columns:
            LABEL_COL = cand
            break

missing_cols = [c for c in [TEXT_COL, LABEL_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Could not find required columns in CSV. Missing: {missing_cols}")

# Drop rows with empty text/label
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# ==== Map labels: vulnerable -> 1, non-vulnerable -> 0 (robust) ====
def map_label(x):
    s = str(x).strip().lower()
    # numeric/boolean passthroughs
    if s in {"1", "true", "yes"}:  return 1
    if s in {"0", "false", "no"}:  return 0

    # common typos/variants
    if s in {"vulnerbale", "vuln", "vul"}: return 1
    if s in {
        "non-vulnerable", "nonvulnerable", "non_vulnerable", "non vuln", "nonvuln",
        "benign", "safe", "clean", "secure", "not vulnerable"
    }: return 0

    # heuristic fallbacks (ordered to avoid matching 'non-vulnerable' as vulnerable)
    if "non" in s and "vul" in s: return 0
    if "vul" in s: return 1

    # last resort: try casting to int
    try:
        v = int(float(s))
        if v in (0, 1): return v
    except Exception:
        pass

    return np.nan  # unknown -> NaN so you can inspect/drop

# apply mapping if not already clean ints
if not np.issubdtype(df[LABEL_COL].dtype, np.number):
    df[LABEL_COL] = df[LABEL_COL].apply(map_label)
else:
    # ensure only 0/1; anything else becomes NaN then dropped
    df[LABEL_COL] = df[LABEL_COL].apply(lambda v: v if v in (0, 1) else np.nan)

# drop rows with unknown labels after mapping
before_n = len(df)
df = df.dropna(subset=[LABEL_COL]).copy()
after_n = len(df)
if after_n < before_n:
    print(f"Dropped {before_n - after_n} rows with unmapped/invalid labels.")

# cast to int
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("Resolved columns ->", {"text_col": TEXT_COL, "label_col": LABEL_COL})
print("Label distribution (0=non-vulnerable, 1=vulnerable):")
print(df[LABEL_COL].value_counts(dropna=False).sort_index())

# ==== HF objects ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==== Dataset / DataLoader ====
class TextClsDS(Dataset):
    def __init__(self, frame, text_col, label_col, tokenizer, max_len=512):
        self.df = frame.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        text = str(self.df.iloc[i][self.text_col])
        label = int(self.df.iloc[i][self.label_col])
        enc = self.tok(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

ds = TextClsDS(df, TEXT_COL, LABEL_COL, tokenizer, MAX_LEN)
dl = DataLoader(ds, batch_size=BATCH_EVAL, shuffle=False, num_workers=0, pin_memory=use_cuda)

# ==== Inference ====
all_labels, all_preds, all_logits = [], [], []

if use_bf16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.bfloat16)
elif use_fp16:
    dtype_ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
else:
    dtype_ctx = nullcontext()

with torch.no_grad():
    if use_cuda:
        torch.cuda.empty_cache()

    with dtype_ctx:
        for batch in tqdm(dl, desc="Predicting"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)
            logits = out.logits.detach().to(torch.float32).cpu().numpy()
            preds = np.argmax(logits, axis=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.tolist())
            all_logits.extend(logits.tolist())

all_labels = np.array(all_labels, dtype=int)
all_preds  = np.array(all_preds, dtype=int)
all_logits = np.array(all_logits, dtype=np.float32)

# Softmax probabilities for class 1
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(all_logits)
prob_1 = probs[:, 1]

# ==== Metrics ====
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)        # sensitivity
f1   = f1_score(all_labels, all_preds, zero_division=0)
mcc  = matthews_corrcoef(all_labels, all_preds)
kap  = cohen_kappa_score(all_labels, all_preds)
mse  = mean_squared_error(all_labels, all_preds)
mae  = mean_absolute_error(all_labels, all_preds)

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

try:
    auc = roc_auc_score(all_labels, prob_1)
except ValueError:
    auc = float("nan")

metrics = {
    "accuracy": acc, "precision": prec, "recall": rec, "sensitivity": rec,
    "specificity": spec, "f1": f1, "mcc": mcc, "kappa": kap,
    "mse": mse, "mae": mae, "roc_auc": auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "n_samples": int(len(all_labels))
}

# ==== Save artifacts ====
metrics_path = os.path.join(EVAL_OUT, "test_metrics_from_saved_model.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

preds_df = df.copy()
preds_df["logit_0"] = all_logits[:, 0]
preds_df["logit_1"] = all_logits[:, 1]
preds_df["prob_0"]  = probs[:, 0]
preds_df["prob_1"]  = probs[:, 1]
preds_df["pred"]    = all_preds
preds_csv = os.path.join(EVAL_OUT, "test_predictions.csv")
preds_df.to_csv(preds_csv, index=False)

print("== Metrics ==")
print(json.dumps(metrics, indent=2))
print(f"\nSaved metrics -> {metrics_path}")
print(f"Saved predictions -> {preds_csv}")
